<a href="https://colab.research.google.com/github/BrianAyekha/-connection-of-MySQL-database-to-NodeJs/blob/main/Week7_Exploring_Pretrained_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7: Exploring Pretrained Models
**Project:** Early fault detection and diagnosis in hospital electromechanical equipment (infant warmer case study)

**Dataset:** AI4I 2020 Predictive Maintenance Dataset (UCI, id 601), used as a stand-in for infant warmer sensor data.

**Author:** Brian Ayekha Mwangala

This notebook is a copy of my Week 5 project notebook (supervised failure classification with Logistic Regression and Random Forest). The new Week 7 discussion on pretrained models is in **Part B**, at the end of the notebook.

# Part A: Existing project work (Week 5)

## 1. Load the dataset

In [ ]:
!pip install -q ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

# Download the dataset and join inputs (features) and targets into one table
ai4i = fetch_ucirepo(id=601)
df = pd.concat([ai4i.data.features, ai4i.data.targets], axis=1)
print("Rows, columns:", df.shape)
df.head()

## 2. Target and features

- **Target (output):** `Machine failure` (0 = normal, 1 = failure)
- **Features (inputs):** `Type`, `Air temperature`, `Process temperature`, `Rotational speed`, `Torque`, `Tool wear`

The failure-mode columns (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) are **not** used as features. They describe *why* a failure happened, so using them would leak the answer. `UID` and `Product ID` are only identifiers.

## 3. Regression or classification?

This is a **classification** problem. The target has only two categories (normal or failure), not a continuous number. The model must decide which class a machine belongs to, and it is not predicting a quantity such as a temperature. More precisely, it is **binary classification**.

In [ ]:
# Define features (X) and target (y)
feature_cols = ["Type", "Air temperature", "Process temperature",
                "Rotational speed", "Torque", "Tool wear"]
X = df[feature_cols]
y = df["Machine failure"]

# Check the class balance
print(y.value_counts())
print(f"\nFailure rate: {y.mean() * 100:.2f}%")

## 4. Train/test split

I use 80% of the data for training and 20% for testing. Only about 3.4% of rows are failures, so I use `stratify=y`. This keeps the same failure rate in both sets, so the test set contains enough failures to evaluate on. `random_state=42` makes the split repeatable.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape, "| failures:", y_train.sum())
print("Test set    :", X_test.shape,  "| failures:", y_test.sum())

## 5. Select and train two models

1. **Logistic Regression:** a simple, interpretable linear baseline. It estimates the probability of failure from a weighted sum of the features.
2. **Random Forest:** an ensemble of many decision trees. It can capture non-linear patterns and interactions between features (for example, high torque combined with high tool wear).

Both models use `class_weight="balanced"`, which gives the rare failure class more weight during training so the models do not simply predict "normal" every time.

**Preprocessing (same for both):** `Type` is categorical, so it is one-hot encoded. The numeric features are scaled. The preprocessing is placed in a pipeline so it is learned from the training data only.

In [ ]:
numeric_cols = ["Air temperature", "Process temperature",
                "Rotational speed", "Torque", "Tool wear"]
categorical_cols = ["Type"]

# One-hot encode Type, scale numeric columns
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(), categorical_cols),
])

# Model 1: Logistic Regression
log_reg = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

# Model 2: Random Forest
rand_forest = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                     random_state=42)),
])

# Train both models on the training data
log_reg.fit(X_train, y_train)
rand_forest.fit(X_train, y_train)
print("Both models trained.")

## 6. Predictions on the test data

In [ ]:
# Predicted classes (0 or 1)
pred_lr = log_reg.predict(X_test)
pred_rf = rand_forest.predict(X_test)

# Predicted probabilities of failure (needed for ROC AUC)
prob_lr = log_reg.predict_proba(X_test)[:, 1]
prob_rf = rand_forest.predict_proba(X_test)[:, 1]

# Show the first few predictions next to the true values
pd.DataFrame({
    "Actual": y_test.values[:10],
    "LogReg": pred_lr[:10],
    "RandomForest": pred_rf[:10],
})

## 7. Evaluation

Accuracy alone is misleading here. About 96.6% of the machines are normal, so a model that always predicts "normal" would score about 96.6% accuracy and detect no failures. I therefore also use:
- **Precision:** of the machines predicted to fail, how many really failed (false alarms lower it)
- **Recall:** of the machines that really failed, how many were caught (missed failures lower it)
- **F1-score:** the balance of precision and recall
- **ROC AUC:** how well the model ranks failures above normal cases across all thresholds
- **Confusion matrix:** the counts of correct and wrong predictions

I also include a "always predict normal" dummy model for reference only. It is not one of the two models.

In [ ]:
def evaluate(name, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred),
        "F1":        f1_score(y_true, y_pred),
        "ROC AUC":   roc_auc_score(y_true, y_prob),
    }

# Reference: always predicts the majority class ("normal")
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred_dummy = dummy.predict(X_test)

results = pd.DataFrame([
    evaluate("Logistic Regression", y_test, pred_lr, prob_lr),
    evaluate("Random Forest",       y_test, pred_rf, prob_rf),
    {"Model": "Dummy (always normal)", "Accuracy": accuracy_score(y_test, pred_dummy),
     "Precision": 0.0, "Recall": 0.0, "F1": 0.0, "ROC AUC": 0.5},
]).set_index("Model")

results.round(3)

In [ ]:
# Detailed report for each model
print("=== Logistic Regression ===")
print(classification_report(y_test, pred_lr, target_names=["Normal", "Failure"], zero_division=0))
print("=== Random Forest ===")
print(classification_report(y_test, pred_rf, target_names=["Normal", "Failure"], zero_division=0))

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, pred) in zip(axes, [("Logistic Regression", pred_lr), ("Random Forest", pred_rf)]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                           display_labels=["Normal", "Failure"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
# Which features did the Random Forest rely on most?
rf_model = rand_forest.named_steps["model"]
feature_names = rand_forest.named_steps["prep"].get_feature_names_out()
importances = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values()

importances.plot(kind="barh", figsize=(7, 4), title="Random Forest feature importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 8. Comparison and discussion

In [ ]:
# Automatic summary of the comparison, based on the actual results above
lr, rf = results.loc["Logistic Regression"], results.loc["Random Forest"]
for metric in ["Accuracy", "Precision", "Recall", "F1", "ROC AUC"]:
    better = "Random Forest" if rf[metric] > lr[metric] else "Logistic Regression"
    print(f"{metric:9s}: LogReg = {lr[metric]:.3f} | RF = {rf[metric]:.3f} -> higher: {better}")

tn, fp, fn, tp = confusion_matrix(y_test, pred_lr).ravel()
print(f"\nLogReg: caught {tp} of {tp+fn} failures, {fp} false alarms")
tn, fp, fn, tp = confusion_matrix(y_test, pred_rf).ravel()
print(f"RF    : caught {tp} of {tp+fn} failures, {fp} false alarms")

**Discussion**

Read this together with the results table and the summary printed above.

- **Accuracy is not enough.** Both models, and even the dummy model, can show high accuracy because failures are rare. Precision, recall and F1 on the *failure* class are the metrics that matter.
- **Typical pattern.** Logistic Regression with balanced weights usually catches many failures (high recall) but raises many false alarms (low precision), because a straight-line boundary cannot describe failures that depend on combinations of features. Random Forest can model these non-linear patterns, so it usually gives higher precision, F1 and ROC AUC. The feature importance plot shows which sensor readings it relied on. I need to confirm in my own table which model actually performed better.
- **Which matters more?** For an infant warmer, a missed failure (false negative) is more dangerous than a false alarm (false positive), so recall carries a lot of weight. Too many false alarms would still make technicians ignore the system, so precision cannot be ignored.
- **Limitations.**
  - The data is synthetic and comes from a milling machine, not from infant warmers.
  - I used a single train/test split and no hyperparameter tuning, so the numbers could change with a different split.
  - The test set contains only a small number of failures, which makes recall and precision noisy.
- **Next steps.** Try cross-validation, tune the models, adjust the decision threshold to favour recall, add an engineered feature such as the difference between process and air temperature, and test on real logged data from my warmer prototype.

---
# Part B: Exploring a Pretrained Model

## 1. Selected model

**Model:** `facebook/bart-large-mnli`

**Link:** https://huggingface.co/facebook/bart-large-mnli

I chose it after browsing the Hugging Face models page filtered by task. This model is listed under **Zero-Shot Classification** and is released under the MIT licence.

## 2. What the model does, its input and its output

**What it is designed to do:** It is the BART-large model after training on the MultiNLI dataset (natural language inference). Natural language inference asks whether one sentence (the premise) supports another sentence (the hypothesis). Researchers (Yin et al., 2019) showed that such a model can be reused for **zero-shot text classification**: the text to classify becomes the premise, and each candidate label becomes a hypothesis such as "This example is a heater failure." The model then scores how well each label fits.

**Input:**
- A piece of English text (for example, a technician's repair note)
- A list of candidate labels that I choose myself at the time of use

**Output:** A score for each candidate label, with the labels ranked from best to worst fit. No retraining is needed to change the labels.

## 3. How it could be useful in my discipline and research area

In biomedical equipment maintenance, a lot of valuable information is stored as **free text**: repair notes, service reports, fault descriptions and work orders. For my project (fault detection and diagnosis in infant warmers), this model could:
- Sort repair notes into fault categories such as "heater or SSR failure", "temperature sensor fault", "power supply fault" and "display or alarm fault".
- Turn old maintenance records into **labelled fault data**, which I currently lack, so that sensor-based models like mine can later be trained on real fault types.
- Help find recurring failure patterns across many service reports.

## 4. How it differs from the machine learning model I have been developing

| Aspect | My project model (Logistic Regression / Random Forest) | `facebook/bart-large-mnli` |
|---|---|---|
| Data type | Numeric sensor readings in a table (temperatures, speed, torque, tool wear) | Natural language text |
| Task | Predict failure or no failure from sensor values | Decide which labels fit a piece of text |
| Training | Trained by me on my own training split | Already trained by others on large text data (BART pretraining, then MultiNLI) |
| Task-specific training | Trained for exactly this one task | None for my task (zero-shot); labels are chosen at use time |
| Size | Small (a few features, simple models) | Large neural network (hundreds of millions of parameters) |
| Interpretability | Coefficients and feature importance can be inspected | Hard to explain why a label got its score |
| How I evaluate it | Test set, with recall, precision and F1 | I would need a hand-labelled set of notes to measure accuracy |

In short, my model learns patterns from the numbers I give it, while the pretrained model brings general language knowledge learned from other data. They work on different kinds of data, so they would complement each other and not compete.

## 5. Optional demonstration

The cell below runs the model on three made-up repair notes. They are invented for illustration and are not real service data. Run it in Colab (the first run downloads the model, which takes a few minutes).

In [ ]:
# Zero-shot classification of (made-up) technician repair notes
!pip install -q transformers

from transformers import pipeline

# Load the pretrained model as a zero-shot classification pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

notes = [
    "Unit not heating. SSR output stuck off. Replaced SSR and tested OK.",
    "Skin probe reads 4 degrees lower than reference thermometer. Replaced probe.",
    "Warmer dead at mains. 12V rail missing. Replaced power supply board.",
]

labels = ["heater or SSR failure", "temperature sensor fault",
          "power supply fault", "display or alarm fault"]

for note in notes:
    result = classifier(note, candidate_labels=labels)
    print("Note:", note)
    for label, score in zip(result["labels"], result["scores"]):
        print(f"   {label:28s} {score:.2f}")
    print()

**Result of my demonstration:** ✏️ *Run the cell above, then replace this line with 1 or 2 sentences: did the top label match what the note describes for each of the three notes? Was the model confident or were the scores close together?*

## 6. Could pretrained models, generative AI or NLP complement my project?

**Yes, on the text side of the project, but not for the core sensor model.**

- **Where they fit:**
  - **Labelling repair notes** with a model like this one could give me fault labels for supervised learning.
  - **Generative AI** could help summarise service manuals and standards and draft documentation.
  - It could also help me plan fault scenarios to simulate on my warmer prototype.
- **Where they do not fit:** The core of my project is detecting faults from numeric sensor readings. A language model does not process these directly, so my supervised and clustering models remain the right tools for that part.
- **Overall:** NLP could help build the dataset and speed up documentation, while the models I built in this course do the fault detection.

## 7. Limitations and precautions before using such a model in science or engineering

- **Not validated for my domain.** The model learned from general English text, not from biomedical repair notes with abbreviations such as "SSR" or "PSU". I would measure its accuracy on notes I have labelled by hand before trusting it.
- **Scores are not true probabilities.** A high score does not guarantee a correct label, and the results can change with the wording of the labels.
- **English only.** Notes written in another language, or in a mix of languages, may not work well.
- **Safety-critical context.** Infant warmers are life-supporting devices. A wrong label in a maintenance decision could have consequences, so a human expert must check the output and the model must not make safety decisions on its own.
- **Privacy.** Repair notes may contain hospital names or patient details. I would remove these before using any online service, and I would prefer running the model locally.
- **Hard to explain.** It is difficult to say why the model gave a particular label, which matters when results must be justified.
- **Bias and unknown training data.** The model may reflect biases in its training text.
- **Reproducibility and cost.** I would record the model version, because it can change, and the model needs substantial memory and computing power.
- **Licence and source.** I would check the model's licence and model card before using it in a published project. This model's card lists the MIT licence.

**Sources**
- Model page: https://huggingface.co/facebook/bart-large-mnli
- BART paper: https://arxiv.org/abs/1910.13461
- Zero-shot classification with NLI models (Yin et al., 2019): https://arxiv.org/abs/1909.00161